# Factor analysis

In [73]:
import itertools
import numpy as np
import pandas as pd

### Configuration constants

In [74]:
DDOF = 1            # degrees of freedom for the standard deviation (n-1)
DEC = 4             # decimals to print
ITERATE = False     # False = a single sweep over the pairs; True = iterate until convergence
MAX_SWEEPS = 100    # only used if ITERATE = True
TOL = 1e-8          # convergence tolerance (angles) if ITERATE = True
OUTPUT_FILE = "resultados.txt"

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda x: f"{x:.{DEC}f}")

### Printing utilities

In [75]:
_LINES = []


def out(text=""):
    """Print to the console and store the line for the output file."""
    text = str(text)
    print(text)
    _LINES.append(text)


def title(text):
    out()
    out("=" * 78)
    out(text)
    out("=" * 78)


def show(name, matrix, rows=None, cols=None):
    """Print a matrix/vector ALWAYS with a title, rounded to DEC decimals."""
    out(f"\n--- {name} ---")
    arr = np.asarray(matrix, dtype=float)
    if arr.ndim == 1:
        arr = arr.reshape(1, -1)
        rows = rows if rows is not None else [""]
    if cols is None:
        cols = [f"c{j + 1}" for j in range(arr.shape[1])]
    out(pd.DataFrame(np.round(arr, DEC), index=rows, columns=cols).to_string())

## Data

In [76]:
data = [
    [1, 85.5, 88.0, 42.1, 45.0, 78.2, 80.5],
    [2, 40.2, 42.5, 91.0, 93.5, 50.1, 48.0],
    [3, 92.0, 90.4, 88.5, 89.0, 95.4, 94.1],
    [4, 35.0, 38.2, 33.1, 35.4, 32.0, 30.5],
    [5, 78.1, 80.0, 45.2, 42.1, 82.4, 85.0],
    [6, 45.0, 43.2, 85.4, 88.0, 40.1, 42.3],
    [7, 89.5, 91.2, 90.1, 92.4, 91.0, 89.5],
    [8, 31.2, 30.0, 88.4, 85.1, 33.5, 35.0],
    [9, 82.0, 85.4, 38.0, 40.2, 80.1, 81.4],
    [10, 50.4, 48.1, 92.5, 94.0, 52.3, 50.1],
]
columns = ["Student", "Y1_Alg1", "Y2_Alg2", "Y3_Log1", "Y4_Log2", "Y5_Mod1", "Y6_Mod2"]
df = pd.DataFrame(data, columns=columns).set_index("Student")
df.index = [f"Student {i}" for i in df.index]

variables = list(df.columns)
students = list(df.index)
X = df.to_numpy(dtype=float)

title("Original Data X (10 students x 6 vars)")
show("Data Matrix X", X, students, variables)


Original Data X (10 students x 6 vars)

--- Data Matrix X ---
            Y1_Alg1  Y2_Alg2  Y3_Log1  Y4_Log2  Y5_Mod1  Y6_Mod2
Student 1   85.5000  88.0000  42.1000  45.0000  78.2000  80.5000
Student 2   40.2000  42.5000  91.0000  93.5000  50.1000  48.0000
Student 3   92.0000  90.4000  88.5000  89.0000  95.4000  94.1000
Student 4   35.0000  38.2000  33.1000  35.4000  32.0000  30.5000
Student 5   78.1000  80.0000  45.2000  42.1000  82.4000  85.0000
Student 6   45.0000  43.2000  85.4000  88.0000  40.1000  42.3000
Student 7   89.5000  91.2000  90.1000  92.4000  91.0000  89.5000
Student 8   31.2000  30.0000  88.4000  85.1000  33.5000  35.0000
Student 9   82.0000  85.4000  38.0000  40.2000  80.1000  81.4000
Student 10  50.4000  48.1000  92.5000  94.0000  52.3000  50.1000


## Step 1. Standard deviation and standardization

In [77]:
title("Step 1. Mean, std and standard matrix Z")
n = X.shape[0]
mean = X.sum(axis=0) / n
# S_i = sqrt( sum((x - mean)^2) / (n - DDOF) )
S = np.sqrt(((X - mean) ** 2).sum(axis=0) / (n - DDOF))
Z = (X - mean) / S
out(f"n = {n} students, p = {X.shape[1]} vars, DDOF = {DDOF}")
show("Means of each variable", mean, ["Mean"], variables)
show("Std S_i", S, ["S_i"], variables)
show("Standard matrix Z = (X - mean) / S_i  (10x6)", Z, students, variables)


Step 1. Mean, std and standard matrix Z
n = 10 students, p = 6 vars, DDOF = 1

--- Means of each variable ---
      Y1_Alg1  Y2_Alg2  Y3_Log1  Y4_Log2  Y5_Mod1  Y6_Mod2
Mean  62.8900  63.7000  69.4300  70.4700  63.5100  63.6400

--- Std S_i ---
     Y1_Alg1  Y2_Alg2  Y3_Log1  Y4_Log2  Y5_Mod1  Y6_Mod2
S_i  24.5769  25.1555  25.9166  25.8831  24.4211  24.6178

--- Standard matrix Z = (X - mean) / S_i  (10x6) ---
            Y1_Alg1  Y2_Alg2  Y3_Log1  Y4_Log2  Y5_Mod1  Y6_Mod2
Student 1    0.9200   0.9660  -1.0545  -0.9840   0.6015   0.6849
Student 2   -0.9232  -0.8428   0.8323   0.8898  -0.5491  -0.6353
Student 3    1.1844   1.0614   0.7358   0.7159   1.3058   1.2373
Student 4   -1.1348  -1.0137  -1.4018  -1.3549  -1.2903  -1.3462
Student 5    0.6189   0.6480  -0.9349  -1.0961   0.7735   0.8677
Student 6   -0.7279  -0.8149   0.6162   0.6773  -0.9586  -0.8669
Student 7    1.0827   1.0932   0.7976   0.8473   1.1257   1.0505
Student 8   -1.2894  -1.3397   0.7320   0.5652  -1.2289  -1.1634

## Step 2. Correlation matrix

In [78]:
title("Step 2. Correlation matrix: R = (1/(n-1)) Z^T Z")
R = (Z.T @ Z) / (n - 1)
show("Correlation matrix: R (6x6)", R, variables, variables)
show("Diagonal: R", np.diag(R), ["diag(R)"], variables)


Step 2. Correlation matrix: R = (1/(n-1)) Z^T Z

--- Correlation matrix: R (6x6) ---
         Y1_Alg1  Y2_Alg2  Y3_Log1  Y4_Log2  Y5_Mod1  Y6_Mod2
Y1_Alg1   1.0000   0.9962  -0.1977  -0.1884   0.9792   0.9840
Y2_Alg2   0.9962   1.0000  -0.2592  -0.2481   0.9742   0.9793
Y3_Log1  -0.1977  -0.2592   1.0000   0.9959  -0.0966  -0.1308
Y4_Log2  -0.1884  -0.2481   0.9959   1.0000  -0.0941  -0.1310
Y5_Mod1   0.9792   0.9742  -0.0966  -0.0941   1.0000   0.9967
Y6_Mod2   0.9840   0.9793  -0.1308  -0.1310   0.9967   1.0000

--- Diagonal: R ---
         Y1_Alg1  Y2_Alg2  Y3_Log1  Y4_Log2  Y5_Mod1  Y6_Mod2
diag(R)   1.0000   1.0000   1.0000   1.0000   1.0000   1.0000


## Step 3. Spectral decomposition + Kaiser criterion

In [79]:
title("Step 3. R = V Lambda V^T")
lam, V = np.linalg.eigh(R)            # eigh returns them in ascending order
order = np.argsort(lam)[::-1]         # largest to smallest
lam, V = lam[order], V[:, order]

# Sign convention (eigenvectors are only defined up to sign):
# make each eigenvector sum to a positive value so results are reproducible.
V = V * np.where(V.sum(axis=0) < 0, -1.0, 1.0)

p = len(lam)
lam_labels = [f"lambda{k + 1}" for k in range(p)]
var_exp = 100 * lam / lam.sum()
var_cum = np.cumsum(var_exp)

show("Eigenvalues", lam, ["lambda"], lam_labels)
show("Eigenvectors V", V, variables, [f"v{k + 1}" for k in range(p)])
show("Explained var (%)", np.vstack([var_exp, var_cum]), ["% explained", "% accumulated"], lam_labels)

# Kaiser criterion: each standardized variable has variance 1, so a factor with
# lambda <= 1 explains less than a single original variable and is discarded.
m = int(np.sum(lam > 1))
out(f"\nKaiser criterion (lambda > 1): kept m = {m} factors.")
for k in range(p):
    status = "Keep" if lam[k] > 1 else "Drop"
    out(f"  lambda{k + 1} = {lam[k]:.{DEC}f}  -> {status}")
out(f"Total variance explained by {m} factors: {var_cum[m - 1]:.{DEC}f} %")


Step 3. R = V Lambda V^T

--- Eigenvalues ---
        lambda1  lambda2  lambda3  lambda4  lambda5  lambda6
lambda   4.0646   1.9014   0.0273   0.0048   0.0013   0.0006

--- Eigenvectors V ---
             v1     v2      v3      v4      v5      v6
Y1_Alg1  0.4897 0.0941  0.5198  0.4014 -0.4756 -0.3063
Y2_Alg2  0.4933 0.0495  0.4488 -0.2685  0.4325  0.5418
Y3_Log1 -0.1604 0.6856 -0.1329  0.4293 -0.1525  0.5281
Y4_Log2 -0.1577 0.6866  0.2415 -0.4189  0.1943 -0.4819
Y5_Mod1  0.4809 0.1657 -0.4845 -0.5116 -0.4876  0.0841
Y6_Mod2  0.4848 0.1404 -0.4665  0.3811  0.5366 -0.3073

--- Explained var (%) ---
               lambda1  lambda2  lambda3  lambda4  lambda5  lambda6
% explained    67.7432  31.6895   0.4545   0.0799   0.0223   0.0107
% accumulated  67.7432  99.4327  99.8872  99.9670  99.9893 100.0000

Kaiser criterion (lambda > 1): kept m = 2 factors.
  lambda1 = 4.0646  -> Keep
  lambda2 = 1.9014  -> Keep
  lambda3 = 0.0273  -> Drop
  lambda4 = 0.0048  -> Drop
  lambda5 = 0.0013  -> Drop

## Step 4. Unrotated loadings

In [80]:
title("Step 4. Loading matrix, no rotation L = V_m * sqrt(Lambda_m)")
L = V[:, :m] * np.sqrt(lam[:m])
h2 = (L ** 2).sum(axis=1)
factors = [f"F{k + 1}" for k in range(m)]
show(f"Loadings: L (6x{m})", L, variables, factors)
show("Communality: h_i^2 = sum_k L_ik^2", h2, ["h_i^2"], variables)


Step 4. Loading matrix, no rotation L = V_m * sqrt(Lambda_m)

--- Loadings: L (6x2) ---
             F1     F2
Y1_Alg1  0.9873 0.1297
Y2_Alg2  0.9945 0.0682
Y3_Log1 -0.3234 0.9454
Y4_Log2 -0.3178 0.9468
Y5_Mod1  0.9694 0.2285
Y6_Mod2  0.9775 0.1936

--- Communality: h_i^2 = sum_k L_ik^2 ---
       Y1_Alg1  Y2_Alg2  Y3_Log1  Y4_Log2  Y5_Mod1  Y6_Mod2
h_i^2   0.9915   0.9937   0.9984   0.9974   0.9920   0.9929


## Step 5. Explanation of the general factor G (text only)

In [81]:
title("Step 5. Why the unrotated solution cannot isolate the specialty (factor G)")
# F1 is the direction of maximum common variance, so it loads on every variable and
# acts as a general factor "g". The solution is also not unique: any orthogonal T
# gives (LT)(LT)^T = L L^T, so we rotate toward a simple structure.
text = """\
In the unrotated space, F1 loads strongly on ALL variables: it is a general
factor 'g' (overall performance). Each student's F1 score is dominated by their
general level, not by their specialty (Algebra/Logic/Modeling), and the other
factors are bipolar (+/-) contrasts that are hard to interpret.
Moreover, the solution is not unique: for any orthogonal T,
(LT)(LT)^T = L T T^T L^T = L L^T, so every rotation reproduces the same R.
That is why we rotate toward a simple structure."""
out(text)
out("\nF1 loads for each var:")
for var, load in zip(variables, L[:, 0]):
    out(f"  {var:8s}: {load: .{DEC}f}")
high = [var for var, load in zip(variables, L[:, 0]) if abs(load) >= 0.5]
out(f"\nMin |load| of F1 = {np.abs(L[:, 0]).min():.{DEC}f}, "
    f"mean = {np.abs(L[:, 0]).mean():.{DEC}f}")
out(f"F1 has a nonzero loading in all {len(variables)} vars "
    f"(|load| >= 0.5 in {len(high)} of them: {', '.join(high)}).")


Step 5. Why the unrotated solution cannot isolate the specialty (factor G)
In the unrotated space, F1 loads strongly on ALL variables: it is a general
factor 'g' (overall performance). Each student's F1 score is dominated by their
general level, not by their specialty (Algebra/Logic/Modeling), and the other
factors are bipolar (+/-) contrasts that are hard to interpret.
Moreover, the solution is not unique: for any orthogonal T,
(LT)(LT)^T = L T T^T L^T = L L^T, so every rotation reproduces the same R.
That is why we rotate toward a simple structure.

F1 loads for each var:
  Y1_Alg1 :  0.9873
  Y2_Alg2 :  0.9945
  Y3_Log1 : -0.3234
  Y4_Log2 : -0.3178
  Y5_Mod1 :  0.9694
  Y6_Mod2 :  0.9775

Min |load| of F1 = 0.3178, mean = 0.7616
F1 has a nonzero loading in all 6 vars (|load| >= 0.5 in 4 of them: Y1_Alg1, Y2_Alg2, Y5_Mod1, Y6_Mod2).


## Step 6. Row normalization (Kaiser normalization)

In [82]:
title("Step 6. Row normalization L_norm[i,:] = L[i,:] / h_i")
h = np.sqrt(h2)                      # stored to denormalize in step 8
L_norm = L / h[:, None]
show("h_i = sqrt(h_i^2) (stored to denormalize)", h, ["h_i"], variables)
show("L_norm (each row has norm 1)", L_norm, variables, factors)


Step 6. Row normalization L_norm[i,:] = L[i,:] / h_i

--- h_i = sqrt(h_i^2) (stored to denormalize) ---
     Y1_Alg1  Y2_Alg2  Y3_Log1  Y4_Log2  Y5_Mod1  Y6_Mod2
h_i   0.9957   0.9969   0.9992   0.9987   0.9960   0.9965

--- L_norm (each row has norm 1) ---
             F1     F2
Y1_Alg1  0.9915 0.1303
Y2_Alg2  0.9977 0.0684
Y3_Log1 -0.3236 0.9462
Y4_Log2 -0.3183 0.9480
Y5_Mod1  0.9733 0.2294
Y6_Mod2  0.9809 0.1943


## Step 7. Pairwise varimax rotation of columns

In [83]:
def optimal_angle(A, B, C, D, p):
    """
    Optimal rotation angle for a pair of columns (formula from class).
        4*theta = arctan2( D - 2AB/p , C - (A^2 - B^2)/p )
    If your version of the formula differs, change it ONLY here.
    """
    num = D - 2 * A * B / p
    den = C - (A ** 2 - B ** 2) / p
    return 0.25 * np.arctan2(num, den)


def rotate_pair(L, i, j):
    """Compute A, B, C, D and theta for columns (i, j) and return the rotated L."""
    p = L.shape[0]
    x, y = L[:, i], L[:, j]
    u = x ** 2 - y ** 2
    v = 2 * x * y
    A = u.sum()
    B = v.sum()
    C = (u ** 2 - v ** 2).sum()
    D = 2 * (u * v).sum()
    theta = optimal_angle(A, B, C, D, p)

    c, s = np.cos(theta), np.sin(theta)
    Rot = np.array([[c, -s], [s, c]])
    L_new = L.copy()
    # [x' y'] = [x y] @ Rot  ->  x' = x cos(theta) + y sin(theta) ; y' = -x sin(theta) + y cos(theta)
    L_new[:, [i, j]] = L[:, [i, j]] @ Rot
    return L_new, Rot, (A, B, C, D, theta)

In [84]:
title("Step 7. Pairwise varimax rotation of columns")
pairs = list(itertools.combinations(range(m), 2))   # (F1,F2), (F1,F3), (F2,F3)
out(f"p = {p} vars, m = {m} factors, pairs: "
    + ", ".join(f"(F{i + 1},F{j + 1})" for i, j in pairs))
out(f"ITERATE = {ITERATE}  ->  " + ("iterate until convergence" if ITERATE else "a single sweep"))

L_rot_norm = L_norm.copy()
T = np.eye(m)              # total accumulated rotation (m x m)
rotations = []             # 2x2 matrices used (for the asserts)

for sweep in range(1, (MAX_SWEEPS if ITERATE else 1) + 1):
    out(f"\n######## Sweep {sweep} ########")
    max_theta = 0.0
    for i, j in pairs:
        L_rot_norm, Rot, (A, B, C, D, theta) = rotate_pair(L_rot_norm, i, j)
        # Accumulate the rotation in T (embedding the 2x2 in an m x m identity)
        G = np.eye(m)
        G[np.ix_([i, j], [i, j])] = Rot
        T = T @ G
        rotations.append(Rot)
        max_theta = max(max_theta, abs(theta))

        out(f"\n>> Pair (F{i + 1}, F{j + 1})")
        out(f"   A = {A:.{DEC}f}   B = {B:.{DEC}f}   C = {C:.{DEC}f}   D = {D:.{DEC}f}")
        out(f"   theta = {theta:.{DEC}f} rad = {np.degrees(theta):.{DEC}f} deg")
        show(f"2x2 rotation matrix for pair (F{i + 1},F{j + 1})", Rot,
             [f"F{i + 1}", f"F{j + 1}"], [f"F{i + 1}'", f"F{j + 1}'"])
        show(f"L_norm after rotating (F{i + 1},F{j + 1})", L_rot_norm, variables, factors)
    if ITERATE and max_theta < TOL:
        out(f"\nConverged in sweep {sweep} (max |theta| = {max_theta:.2e}).")
        break

show("Accumulated rotation matrix T (L_rot_norm = L_norm @ T)", T, factors, factors)


Step 7. Pairwise varimax rotation of columns
p = 6 vars, m = 2 factors, pairs: (F1,F2)
ITERATE = False  ->  a single sweep

######## Sweep 1 ########

>> Pair (F1, F2)
   A = 2.1880   B = 0.0068   C = 3.6614   D = 4.2042
   theta = 0.2431 rad = 13.9273 deg

--- 2x2 rotation matrix for pair (F1,F2) ---
      F1'     F2'
F1 0.9706 -0.2407
F2 0.2407  0.9706

--- L_norm after rotating (F1,F2) ---
             F1      F2
Y1_Alg1  0.9937 -0.1122
Y2_Alg2  0.9848 -0.1737
Y3_Log1 -0.0864  0.9963
Y4_Log2 -0.0807  0.9967
Y5_Mod1  0.9999 -0.0116
Y6_Mod2  0.9989 -0.0475

--- Accumulated rotation matrix T (L_rot_norm = L_norm @ T) ---
       F1      F2
F1 0.9706 -0.2407
F2 0.2407  0.9706


## Step 8. Clean loadings L*

In [85]:
title("Step 8. Clean loadings L*[i,:] = L_rot_norm[i,:] * h_i")
L_star = L_rot_norm * h[:, None]
h2_star = (L_star ** 2).sum(axis=1)
show("Rotated loadings L*", L_star, variables, factors)
show("Communality comparison", np.vstack([h2, h2_star]), ["h_i^2 of L", "h_i^2 of L*"], variables)
out(f"Communalities of L* = communalities of L?  {np.allclose(h2, h2_star)}")


Step 8. Clean loadings L*[i,:] = L_rot_norm[i,:] * h_i

--- Rotated loadings L* ---
             F1      F2
Y1_Alg1  0.9895 -0.1117
Y2_Alg2  0.9817 -0.1732
Y3_Log1 -0.0863  0.9955
Y4_Log2 -0.0806  0.9954
Y5_Mod1  0.9959 -0.0116
Y6_Mod2  0.9953 -0.0474

--- Communality comparison ---
             Y1_Alg1  Y2_Alg2  Y3_Log1  Y4_Log2  Y5_Mod1  Y6_Mod2
h_i^2 of L    0.9915   0.9937   0.9984   0.9974   0.9920   0.9929
h_i^2 of L*   0.9915   0.9937   0.9984   0.9974   0.9920   0.9929
Communalities of L* = communalities of L?  True


## Step 9. Reproduced theoretical matrix

In [86]:
title("Step 9. Reproduced theoretical matrix Sigma_star = L* L*^T")
Sigma_star = L_star @ L_star.T
show("Sigma_star = L* L*^T (standardized / correlation scale)", Sigma_star, variables, variables)
D_S = np.diag(S)
Sigma_cov = D_S @ Sigma_star @ D_S
show("Sigma_cov = D_S Sigma_star D_S (theoretical covariance, original scale)",
     Sigma_cov, variables, variables)


Step 9. Reproduced theoretical matrix Sigma_star = L* L*^T

--- Sigma_star = L* L*^T (standardized / correlation scale) ---
         Y1_Alg1  Y2_Alg2  Y3_Log1  Y4_Log2  Y5_Mod1  Y6_Mod2
Y1_Alg1   0.9915   0.9907  -0.1966  -0.1910   0.9867   0.9901
Y2_Alg2   0.9907   0.9937  -0.2571  -0.2515   0.9797   0.9853
Y3_Log1  -0.1966  -0.2571   0.9984   0.9979  -0.0975  -0.1330
Y4_Log2  -0.1910  -0.2515   0.9979   0.9974  -0.0918  -0.1274
Y5_Mod1   0.9867   0.9797  -0.0975  -0.0918   0.9920   0.9918
Y6_Mod2   0.9901   0.9853  -0.1330  -0.1274   0.9918   0.9929

--- Sigma_cov = D_S Sigma_star D_S (theoretical covariance, original scale) ---
          Y1_Alg1   Y2_Alg2   Y3_Log1   Y4_Log2  Y5_Mod1  Y6_Mod2
Y1_Alg1  598.8902  612.4902 -125.2245 -121.4750 592.2296 599.0542
Y2_Alg2  612.4902  628.8284 -167.6349 -163.7727 601.8631 610.1791
Y3_Log1 -125.2245 -167.6349  670.6132  669.3813 -61.6933 -84.8853
Y4_Log2 -121.4750 -163.7727  669.3813  668.1732 -58.0287 -81.1629
Y5_Mod1  592.2296  601.8631  -

## Step 10. Error matrix

In [87]:
title("Step 10. Error matrix E = R - Sigma_star")
E = R - Sigma_star
show("Error matrix E", E, variables, variables)
frob = np.sqrt((E ** 2).sum())
off_diag = E[~np.eye(E.shape[0], dtype=bool)]
out(f"Frobenius norm ||E||_F = {frob:.{DEC}f}")
out(f"Max |error| off the diagonal = {np.abs(off_diag).max():.{DEC}f}")
out("(The diagonal of E is the uniqueness 1 - h_i^2 of each variable.)")


Step 10. Error matrix E = R - Sigma_star

--- Error matrix E ---
         Y1_Alg1  Y2_Alg2  Y3_Log1  Y4_Log2  Y5_Mod1  Y6_Mod2
Y1_Alg1   0.0085   0.0055  -0.0011   0.0026  -0.0076  -0.0062
Y2_Alg2   0.0055   0.0063  -0.0021   0.0034  -0.0055  -0.0060
Y3_Log1  -0.0011  -0.0021   0.0016  -0.0019   0.0008   0.0023
Y4_Log2   0.0026   0.0034  -0.0019   0.0026  -0.0023  -0.0036
Y5_Mod1  -0.0076  -0.0055   0.0008  -0.0023   0.0080   0.0049
Y6_Mod2  -0.0062  -0.0060   0.0023  -0.0036   0.0049   0.0071
Frobenius norm ||E||_F = 0.0277
Max |error| off the diagonal = 0.0076
(The diagonal of E is the uniqueness 1 - h_i^2 of each variable.)


## Step 11. Factor scores (Thompson / regression)

In [88]:
title("Step 11. Factor scores (Thompson/regression) F = Z R^(-1) L*")
F = Z @ np.linalg.inv(R) @ L_star
show(f"Factor scores F (10x{m})", F, students, factors)


areas = {"Alg": "Algebra", "Log": "Logic", "Mod": "Modeling"}
out("\nInterpretation of the rotated factors (|load| >= 0.5):")
for k in range(m):
    strong = [var for var, load in zip(variables, L_star[:, k]) if abs(load) >= 0.5]
    name = " + ".join(a for key, a in areas.items() if any(key in var for var in strong)) or "?"
    out(f"  F{k + 1} = {name:9s} <- variables: {', '.join(strong)}")


Step 11. Factor scores (Thompson/regression) F = Z R^(-1) L*

--- Factor scores F (10x2) ---
                F1      F2
Student 1   0.7150 -0.9757
Student 2  -0.6726  0.8190
Student 3   1.2818  0.8403
Student 4  -1.3380 -1.4997
Student 5   0.6529 -0.9489
Student 6  -0.8014  0.5710
Student 7   1.1773  0.9174
Student 8  -1.2190  0.5529
Student 9   0.6685 -1.1407
Student 10 -0.4644  0.8644

Interpretation of the rotated factors (|load| >= 0.5):
  F1 = Algebra + Modeling <- variables: Y1_Alg1, Y2_Alg2, Y5_Mod1, Y6_Mod2
  F2 = Logic     <- variables: Y3_Log1, Y4_Log2
